In [1]:
import os
import numpy as np
import cv2
import json
from scipy.io import loadmat
import h5py
import pandas as pd
from tifffile import imread
import matplotlib.pyplot as plt
import copy
from extract_features_functions import *

In [2]:
# PUT PATH TO NDPI IMAGES HERE
# pth_WSI_ndpi = r'\\10.99.68.178\andreex\data\monkey fetus\gestational 40'
pth_WSI_ndpi = r'\\10.162.80.16\Andre\data\monkey fetus\bissected_monkey_GS55'

# CREATES FOLDERS
pth_segmentation_json = os.path.join(pth_WSI_ndpi,'StarDist_12_12_23', 'json')
# pth_crop_data = r'\\10.99.68.178\andreex\data\monkey fetus\gestational 40\2_5x\cropped_images\bounding_boxes'
pth_pixel_res = os.path.join(pth_WSI_ndpi,'segmentation_analysis', 'pix_res_info')
outpth = os.path.join(pth_WSI_ndpi,'StarDist_12_12_23', 'stardist_feature_df_pickles')
os.makedirs(outpth, exist_ok=True)

# Get the files
WSIs = get_sorted_files(pth_WSI_ndpi, '.ndpi')
jsons = get_sorted_files(pth_segmentation_json, '.json')
pixel_res_files = get_sorted_files(pth_pixel_res, '.mat')
# crop_mats = get_sorted_files(pth_crop_data, '.mat')

# Display first files and counts
if WSIs: print("First WSI:", WSIs[0])
if jsons: print("First JSON:", jsons[0])
if pixel_res_files: print("First Pixel Res File:", pixel_res_files[0])

print(f"Number of WSIs: {len(WSIs)}")
print(f"Number of JSON files: {len(jsons)}")
print(f"Number of Pixel Res Files: {len(pixel_res_files)}")

First WSI: \\10.162.80.16\Andre\data\monkey fetus\bissected_monkey_GS55\monkey_fetus_GS55_0001.ndpi
First JSON: \\10.162.80.16\Andre\data\monkey fetus\bissected_monkey_GS55\StarDist_12_12_23\json\monkey_fetus_GS55_0001.json
First Pixel Res File: \\10.162.80.16\Andre\data\monkey fetus\bissected_monkey_GS55\segmentation_analysis\pix_res_info\monkey_fetus_GS55_0001.mat
Number of WSIs: 116
Number of JSON files: 116
Number of Pixel Res Files: 116


In [3]:
if check_file_alignment(jsons, WSIs, pixel_res_files):
    print("All files match.")
else:
    print("There is a mismatch in the files.")

All files match.


In [4]:
for i, json_f_name in enumerate(jsons):
    
    nm = json_f_name.split('\\')[-1].split('.')[0]
    
    outnm = os.path.join(outpth, f'{nm}.pkl')
    print(outnm)
    
    if not os.path.exists(outnm):
        
        # delete, just for testing
        #i = 500
        #json_f_name = jsons[500]
        
        HE_20x_WSI = imread(WSIs[i])
        
        print(WSIs[i])
        print(json_f_name)
        
        try:
            segmentation_data = json.load(open(json_f_name))
        except:
            print(f'error reading json... Skipping {json_f_name}')
            continue
    
        centroids = [nuc['centroid'][0] for nuc in segmentation_data]
        contours = [nuc['contour'] for nuc in segmentation_data]
        contours_fixed = fix_contours(contours)
        
        offset = 30  # radius of image that gets cropped from WSI, used for getting rgb intensity average inside nuc contour
        
        centroids_np = np.array(centroids)  # for other formatting
        contours_np = np.array(contours)
        
        r_avg_list = []
        g_avg_list = []
        b_avg_list = []
        
        areas = []
        perimeters = []
        circularities = []
        aspect_ratios = []
        image_ids = []
        classes = []
        
        compactness_a, eccentricity_a, euler_number_a, extent_a, form_factor_a, maximum_radius_a, mean_radius_a, median_radius_a, minor_axis_length_a, major_axis_length_a, orientation_degrees_a = [], [], [], [], [], [], [], [], [], [], []
        
        np_centroids = np.array(centroids)
        
        for j in range(len(contours_fixed)):
            #break
            
            centroid = centroids[j]
            # print(f'centroid: {centroid}')
            contour_raw = copy.copy(contours_fixed[j])  # used for rgb intensities
            
            # get rbg intensity averages
            r_avg, g_avg, b_avg = get_rbg_avg(centroid, contour_raw, offset, HE_20x_WSI)
            # print(r_avg, g_avg, b_avg)
            
            r_avg_list.append(r_avg)
            g_avg_list.append(g_avg)
            b_avg_list.append(b_avg)
            
            contour = contours_np[j][0].transpose()  # used for other stuff, too lazy to make formatting the same
            area = cntarea(contour)
            perimeter = cntperi(contour)
            circularity = 4 * np.pi * area / perimeter ** 2
            MA = cntMA(contour)
            [MA, ma, orientation] = MA
            aspect_ratio = MA / ma
            #center_x = centroid[0]
            #center_y = centroid[1]
            
            cent_x = np_centroids[j,0]
            cent_y = np_centroids[j,1]
            
            #compactness and form_factor are stupid because they are basically same as circularity, maybe extent too
            
            compactness = perimeter ** 2 / area
            eccentricity = np.sqrt(1 - (ma / MA) ** 2)
            extent = area / (MA * ma)
            form_factor = (perimeter ** 2) / (4 * np.pi * area)
            major_axis_length = MA
            maximum_radius = np.max(np.linalg.norm(contour - centroid, axis=1))
            mean_radius = np.mean(np.linalg.norm(contour - centroid, axis=1))
            median_radius = np.median(np.linalg.norm(contour - centroid, axis=1))
            minor_axis_length = ma
            orientation_degrees = np.degrees(orientation)
            
            areas.append(area)
            perimeters.append(perimeter)
            circularities.append(circularity)
            aspect_ratios.append(aspect_ratio)
    
            # additional features
            compactness_a.append(compactness)
            eccentricity_a.append(eccentricity)
            extent_a.append(extent)
            form_factor_a.append(form_factor)
            maximum_radius_a.append(maximum_radius)
            mean_radius_a.append(mean_radius)
            median_radius_a.append(median_radius)
            minor_axis_length_a.append(minor_axis_length)
            major_axis_length_a.append(major_axis_length)
            orientation_degrees_a.append(orientation_degrees)
            
            
        # exit loop
            
        dat = {
            'Centroid_x': np_centroids[:,1],
            'Centroid_y': np_centroids[:,0],
            'Area': areas,
            'Perimeter': perimeters,
            'Circularity': circularities,
            'Aspect Ratio': aspect_ratios,
            'compactness' : compactness_a,
            'eccentricity' : eccentricity_a,
            'extent' : extent_a,
            'form_factor' : form_factor_a,
            'maximum_radius' : maximum_radius_a,
            'mean_radius' : mean_radius_a,
            'median_radius' : median_radius_a,
            'minor_axis_length' : minor_axis_length_a,
            'major_axis_length' : major_axis_length_a,
            'orientation_degrees' : orientation_degrees_a,
            'r_mean_intensity' : r_avg_list,
            'g_mean_intensity' : g_avg_list,
            'b_mean_intensity' : b_avg_list,
            'slide_num': nm[-4:]  # fix this for your own needs, this gets slide number for my monkey fetus
        }
    
        df = pd.DataFrame(dat).astype(np.float32)  # save a little space with float16 type -> Edit 2 months later, this did not save time.
        
        df.to_pickle(outnm)
        #break
    else:
        print('skipping')


\\10.162.80.16\Andre\data\monkey fetus\bissected_monkey_GS55\StarDist_12_12_23\stardist_feature_df_pickles\monkey_fetus_GS55_0001.pkl
skipping
\\10.162.80.16\Andre\data\monkey fetus\bissected_monkey_GS55\StarDist_12_12_23\stardist_feature_df_pickles\monkey_fetus_GS55_0004.pkl
skipping
\\10.162.80.16\Andre\data\monkey fetus\bissected_monkey_GS55\StarDist_12_12_23\stardist_feature_df_pickles\monkey_fetus_GS55_0007.pkl
skipping
\\10.162.80.16\Andre\data\monkey fetus\bissected_monkey_GS55\StarDist_12_12_23\stardist_feature_df_pickles\monkey_fetus_GS55_0010.pkl
skipping
\\10.162.80.16\Andre\data\monkey fetus\bissected_monkey_GS55\StarDist_12_12_23\stardist_feature_df_pickles\monkey_fetus_GS55_0013.pkl
skipping
\\10.162.80.16\Andre\data\monkey fetus\bissected_monkey_GS55\StarDist_12_12_23\stardist_feature_df_pickles\monkey_fetus_GS55_0016.pkl
skipping
\\10.162.80.16\Andre\data\monkey fetus\bissected_monkey_GS55\StarDist_12_12_23\stardist_feature_df_pickles\monkey_fetus_GS55_0019.pkl
skipping

In [5]:
print(len(df))
print(len(centroids))

NameError: name 'df' is not defined

In [19]:
df

,Centroid_x,Centroid_y,Area,Perimeter,Circularity,Aspect Ratio,compactness,eccentricity,extent,form_factor,maximum_radius,mean_radius,median_radius,minor_axis_length,major_axis_length,orientation_degrees,r_mean_intensity,g_mean_intensity,b_mean_intensity,slide_num
0,2174.0,806.0,88.804222,35.844608,0.868552,1.420033,14.468186,0.709993,0.762761,1.151342,7.550079,5.251587,5.106949,9.054685,12.857948,68.559776,173.360001,131.889999,169.550003,346.0
1,2178.0,840.0,50.527615,27.985064,0.810748,1.252869,15.499719,0.602435,0.752481,1.233428,4.819451,3.997113,4.048697,7.320889,9.172114,4651.330566,166.330002,129.300003,165.789993,346.0
2,2178.0,848.0,77.828651,34.006004,0.845743,1.173202,14.858387,0.522942,0.751323,1.182393,6.725778,4.898816,4.940513,9.396585,11.024095,4596.118652,175.160004,132.429993,166.089996,346.0
3,6584.0,1200.0,78.097298,33.166382,0.892174,1.026910,14.085107,0.227426,0.768620,1.120857,6.428418,4.956306,4.979295,9.947092,10.214766,9025.058594,130.190002,94.730003,140.210007,346.0
4,6488.0,1176.0,78.823692,33.133999,0.902233,1.061564,13.928069,0.335594,0.772268,1.108361,5.707822,5.010150,5.143089,9.805538,10.409203,2865.539795,164.089996,120.699997,164.490005,346.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33788,24586.0,18822.0,42.539520,27.462839,0.708780,2.510163,17.729574,0.917220,0.756768,1.410875,6.928905,3.457972,3.026257,4.732209,11.878614,6355.478516,177.490005,153.669998,181.679993,346.0
33789,25460.0,19776.0,103.025314,40.006786,0.808885,1.567182,15.535433,0.769963,0.735638,1.236270,9.006705,5.623678,5.646786,9.453229,14.814926,2224.104492,155.220001,148.089996,153.699997,346.0
33790,24454.0,19334.0,34.997208,27.313126,0.589523,2.874562,21.316183,0.937539,0.746794,1.696288,6.441025,3.035988,2.873910,4.037666,11.606522,5287.464355,170.919998,147.820007,179.309998,346.0
33791,24618.0,19262.0,46.782738,31.960157,0.575543,3.073775,21.833944,0.945600,0.739665,1.737490,6.823304,3.550376,3.071185,4.536169,13.943161,3744.861328,164.589996,123.360001,170.789993,346.0
